In [ ]:
# -*- coding: utf-8 -*-
"""
H2B2A7-INTEGRACION-00 | UNA SOLA CELDA | AUDITORIA REPRODUCIBLE DEL CLASIFICADOR.

NO QPU, NO QISKIT, NO ENTRENAMIENTO, NO REOPTIMIZACION, NO CAMBIO DE UMBRALES.
Se identifican los artefactos originales de H2B2A0/A2/A4/A5/A7.
Solo se reconstruyen scores cuando los checkpoints y la salida archivada
permiten una comparacion exacta. No se selecciona ganador con OUTER HOLD.

Requisitos: Python 3.10+, numpy. Los CSV se procesan con csv estandar.
Si no estan todos los checkpoints, NO se inventan resultados: se informa PENDIENTE.
"""
from pathlib import Path
from datetime import datetime, timezone
from itertools import product
import collections
import csv
import hashlib
import json
import os
import platform
import sys
import traceback

import numpy as np

# ===================== CONFIGURACION EDITABLE =====================
ROOT_OVERRIDE = None   # Ejemplo: r'C:\Users\javie\OneDrive\Escritorio\Tesis4'
REPLAY_ARCHIVED_OUTER = True  # Repite los scores YA ARCHIVADOS, sin entrenar ni ajustar.
TOL_SCORE_MAX = 5e-5  # Tolerancia estricta en el replay de score float.
MAX_CANDIDATE_FILES = 12000
# ================================================================

VERSION = 'H2B2A7-INTEGRACION-00-v1.2'
UTC = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SCORE_ARTIFACTS = {
    'TEACHER_R3': 'ARCH4FH2B2A0_FREEZE_R3_TEACHER.npz',
    'HEAD4': 'ARCH4FH2B2A2_BEST_JOINT_FOUR_HEAD_MODEL.npz',
    'SAFE_RANK2': 'ARCH4FH2B2A4_1D2_SAFE_LOCAL_MODEL.npz',
    'X6_PAULI': 'ARCH4FH2B1D2_X6_PAULI_EXPECTATIONS.npy',
    'QWC24': 'MODELO_FINAL_CONGELADO.npz',
    'QWC24_CONTROL': 'congelado_K24_Ridge_var.npz',
    'QWC64': 'elegido_N10000.npz',
    'QWC_ORIG': 'modelo_QWC_G64.npz',
    'OUTER_SCORES': 'predicciones_OUTER_HOLD.csv',
    'OUTER_REPORT': 'REPORTE_FINAL.json',
    'OUTER_MANIFEST': 'MANIFEST_PREVIO.json',
    'SAFE_ALGEBRA': 'coeficientes_algebraicos_congelados.npz',
    'QWC365_COVERAGE': 'certificado_cobertura.json',
    'QWC365_BASES': 'bases_CONSERVAR_24.npz',
    'QWC_SHOTS_REFERENCE': 'asignacion_referencia_100k_10k.npz',
    'STUDENT_COMP': 'ARCH4FH2B2A5_1E_RANKING_SAFE_MODEL.npz',
    'STUDENT_COMP_QPY': 'ARCH4FH2B2A5_1E_RANKING_SAFE_CIRCUITS.qpy',
    'STUDENT_COMP_REPORT': 'ARCH4FH2B2A5_1E_RANKING_SAFE_SUMMARY.csv',
    'STUDENT_PARENT_1C': 'ARCH4FH2B2A5_1C_FIXED48_MATURE_MODEL.npz',
    'SOURCE_NOTEBOOK': 'ContinuacionnuevaModelosdataset2(20261105-030316).ipynb',
}
# Rutas relativas al ROOT verificadas contra el codigo fuente H2B2A7-13/14/15/16.
RELATIVE_HINTS = {
    'QWC24': 'H2B2A7_PAULI_AUDIT/H2B2A7_12_LOCAL_QWC_SWAPS/MODELO_FINAL_CONGELADO.npz',
    'QWC24_CONTROL': 'H2B2A7_PAULI_AUDIT/H2B2A7_10_QWC_FULL_PAULI/congelado_K24_Ridge_var.npz',
    'QWC64': 'H2B2A7_PAULI_AUDIT/H2B2A7_7_SHOT_BUDGET_SELECTION/elegido_N10000.npz',
    'QWC_ORIG': 'H2B2A7_PAULI_AUDIT/H2B2A7_3_GROUP_QWC/modelo_QWC_G64.npz',
    'OUTER_SCORES': 'H2B2A7_PAULI_AUDIT/H2B2A7_13_FINAL_OUTER_HOLD/predicciones_OUTER_HOLD.csv',
    'OUTER_REPORT': 'H2B2A7_PAULI_AUDIT/H2B2A7_13_FINAL_OUTER_HOLD/REPORTE_FINAL.json',
    'OUTER_MANIFEST': 'H2B2A7_PAULI_AUDIT/H2B2A7_13_FINAL_OUTER_HOLD/MANIFEST_PREVIO.json',
    'SAFE_ALGEBRA': 'H2B2A7_PAULI_AUDIT/H2B2A7_14_AUDITORIA_RECURSOS/coeficientes_algebraicos_congelados.npz',
    'QWC365_COVERAGE': 'H2B2A7_PAULI_AUDIT/H2B2A7_15_COBERTURA_QWC_OPTIMA/certificado_cobertura.json',
    'QWC365_BASES': 'H2B2A7_PAULI_AUDIT/H2B2A7_15_COBERTURA_QWC_OPTIMA/bases_CONSERVAR_24.npz',
    'QWC_SHOTS_REFERENCE': 'H2B2A7_PAULI_AUDIT/H2B2A7_16_VARIANZA_Y_SHOTS/asignacion_referencia_100k_10k.npz',
}
EXPECTED_KEYS = {
    'TEACHER_R3': ['baseline','lambdas','t_target','s3','y','inner_train_idx','inner_val_idx','outer_hold_idx'],
    'HEAD4': ['A4','c4','head_coefficients','Z4_train','Z4_val'],
    'SAFE_RANK2': ['W','d','B','C','T_train','T_val','znew_train','znew_val'],
    'QWC24': ['theta','basis_ids','pauli_index','shots_por_base','threshold_fit','global_val'],
    'QWC24_CONTROL':['theta','basis_ids','shots_por_base','threshold_fit'],
    'QWC64':['theta','basis_ids','pauli_index','shots_per_base','threshold_fit','global_val'],
    'QWC_ORIG':['coef_pauli_original'],
    'SAFE_ALGEBRA':['Q','E','Fmap','a'],
    'QWC_SHOTS_REFERENCE':['basis_ids','n6_por_base','n4_por_canal','costos_copias'],
    'STUDENT_COMP':['local_angles','rxx_angles','diagonal'],
}


def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as fh:
        for block in iter(lambda:fh.read(1<<20),b''):
            h.update(block)
    return h.hexdigest()


def json_save(path,data):
    path.write_text(json.dumps(data,indent=2,ensure_ascii=False,default=str),encoding='utf-8')


def csv_save(path, rows, columns=None):
    rows=list(rows)
    columns=columns or (list(rows[0]) if rows else ['nota'])
    with open(path,'w',newline='',encoding='utf-8-sig') as fh:
        w=csv.DictWriter(fh,fieldnames=columns,extrasaction='ignore')
        w.writeheader()
        for row in rows:w.writerow(row)


def csv_read(path):
    with open(path,'r',encoding='utf-8-sig',newline='') as fh:
        return list(csv.DictReader(fh))


def choose_root():
    here=Path.cwd().resolve()
    cand=[]
    if ROOT_OVERRIDE:cand.append(Path(ROOT_OVERRIDE))
    cand.extend([Path(r'C:\Users\javie\OneDrive\Escritorio\Tesis4'),
                 Path.home()/'OneDrive'/'Escritorio'/'Tesis4',
                 Path.home()/'OneDrive'/'Desktop'/'Tesis4',
                 Path.home()/'Desktop'/'Tesis4',
                 here, *list(here.parents)[:3]])
    cand.extend([p.parent for p in list(here.glob('**/ARCH4FH2B2A0_FREEZE_R3_TEACHER.npz'))[:2]] if len(str(here))<80 else [])
    scored=[];seen=set()
    for p in cand:
        try:p=p.resolve()
        except Exception:continue
        if p in seen or not p.is_dir():continue
        seen.add(p)
        # Si estamos en la subcarpeta AUDIT, la raiz real es el padre.
        if p.name=='H2B2A7_PAULI_AUDIT':p=p.parent
        count=sum((p/n).exists() for n in ['ARCH4FH2B2A0_FREEZE_R3_TEACHER.npz',
               'ARCH4FH2B2A4_1D2_SAFE_LOCAL_MODEL.npz', 'ARCH4FH2B1D2_X6_PAULI_EXPECTATIONS.npy'])
        count+=int((p/'H2B2A7_PAULI_AUDIT').is_dir())
        scored.append((count,p))
    if not scored:raise RuntimeError('No encuentro un directorio de trabajo accesible.')
    scored.sort(key=lambda a:(a[0],len(str(a[1]))),reverse=True)
    best=scored[0][1]
    return best


ROOT=choose_root()
AUDIT=ROOT/'H2B2A7_PAULI_AUDIT'
AUDIT.mkdir(exist_ok=True,parents=True)
OUT=AUDIT/f'H2B2A7_INTEGRACION_00_AUDITORIA_DOS_RUTAS_{UTC}'
OUT.mkdir(parents=True,exist_ok=False)
print('='*99)
print('H2B2A7-INTEGRACION-00 | AUDITORIA SIN ENTRENAMIENTO NI IBM | DOS ARQUITECTURAS')
print('ROOT:',ROOT)
print('SALIDA:',OUT)
print('='*99)

# Inventario sin confiar en coincidencias parciales de archivos homonimos.
lookup={}
allfound=collections.defaultdict(list)
for key,name in SCORE_ARTIFACTS.items():
    if key in RELATIVE_HINTS:
        p=ROOT/RELATIVE_HINTS[key]
        if p.is_file():allfound[key].append(p)
    p=ROOT/name
    if p.is_file() and p not in allfound[key]:allfound[key].append(p)
# Solo recurrimos a una busqueda acotada para archivos ausentes.
missing_keys=[k for k in SCORE_ARTIFACTS if not allfound[k]]
if missing_keys:
    targets={SCORE_ARTIFACTS[k] for k in missing_keys}
    skip={'.git','.venv','venv','__pycache__','.ipynb_checkpoints','node_modules'}
    n_seen=0
    for directory,dirs,files in os.walk(ROOT):
        dirs[:]=[d for d in dirs if d not in skip and not d.startswith('H2B2A7_INTEGRACION_00_AUDITORIA')]
        n_seen+=len(files)
        if n_seen>MAX_CANDIDATE_FILES:break
        for name in set(files)&targets:
            f=Path(directory)/name
            for k in missing_keys:
                if SCORE_ARTIFACTS[k]==name and f not in allfound[k]:allfound[k].append(f)
for k in SCORE_ARTIFACTS:
    candidates=allfound[k]
    if k in RELATIVE_HINTS:
        wanted=ROOT/RELATIVE_HINTS[k]
        if wanted in candidates:candidates=[wanted]+[v for v in candidates if v!=wanted]
    lookup[k]=candidates[0] if candidates else None

checks=[];files_verified=[];read_errors=[]
for k,name in SCORE_ARTIFACTS.items():
    path=lookup[k]
    row=dict(clave=k,archivo=name,estado='NO_ENCONTRADO',ruta='',sha256='',tamano_bytes='',
             claves_npz='',claves_faltantes='',duplicados=len(allfound[k]))
    if path:
        row.update(estado='ENCONTRADO',ruta=str(path),sha256=sha256(path),tamano_bytes=path.stat().st_size)
        files_verified.append(path)
        if path.suffix.lower()=='.npz':
            try:
                with np.load(path,allow_pickle=False) as z:
                    keys=set(z.files)
                    bad=sorted(set(EXPECTED_KEYS.get(k,[]))-keys)
                    row['claves_npz']=';'.join(sorted(keys))[:1000]
                    row['claves_faltantes']=';'.join(bad)
                    if bad:row['estado']='CLAVES_INCOMPLETAS'
            except Exception as ex:
                row['estado']='NPZ_INACCESIBLE';read_errors.append(f'{k}: {ex}')
    checks.append(row)

def get_npz(key,keys):
    p=lookup.get(key)
    if not p or not p.is_file():raise FileNotFoundError(f'{key}: checkpoint ausente')
    with np.load(p,allow_pickle=False) as z:
        bad=set(keys)-set(z.files)
        if bad:raise KeyError(f'{key}: claves faltantes: {sorted(bad)}')
        return {v:np.array(z[v]) for v in keys}

# Comprobamos integridad contra el manifiesto PREVIO a la apertura historica de OUTER HOLD.
manifest_status='NO_DISPONIBLE';manifest_mismatches=[]
if lookup['OUTER_MANIFEST']:
    try:
        man=json.loads(lookup['OUTER_MANIFEST'].read_text(encoding='utf-8'))
        prev=man.get('hashes',{})
        if not prev:manifest_status='SIN_HASHES_COMPARABLES'
        else:
            for k in ['TEACHER_R3','SAFE_RANK2','HEAD4','QWC24','QWC24_CONTROL','QWC64','QWC_ORIG']:
                p=lookup[k]
                if p and p.name in prev and sha256(p)!=prev[p.name]:
                    manifest_mismatches.append(k)
            manifest_status='COINCIDE' if not manifest_mismatches else 'DIVERGENCIA_SHA256'
    except Exception as ex:
        manifest_status='MANIFIESTO_ILEGIBLE';read_errors.append('MANIFEST: '+str(ex))
if manifest_mismatches:print('ALERTA: divergencia con manifiesto OUTER previo:',manifest_mismatches)

# ================= REPLAY DE SCORE TEACHER (sin datos externos) =================
replays=[];score_rows=[];replay_details={}
teacher=None;teacher_reconstruction=None
try:
    t=get_npz('TEACHER_R3',EXPECTED_KEYS['TEACHER_R3'])
    baseline=np.asarray(t['baseline'],float).ravel()
    lam=np.asarray(t['lambdas'],float)
    lat=np.asarray(t['t_target'],float)
    stored=np.asarray(t['s3'],float).ravel()
    assert baseline.ndim==1 and lam.shape==lat.shape==(len(baseline),3)
    assert stored.shape==baseline.shape and np.isfinite(lam).all()
    teacher_reconstruction=baseline+np.einsum('ij,ij->i',lam,lat)
    err=float(np.max(np.abs(teacher_reconstruction-stored)))
    status='REPRODUCIDO' if err<=1e-8 else 'DIVERGENCIA_NUMERICA'
    replays.append(dict(arquitectura='TEACHER_R3',split='ARCHIVO_COMPLETO',estado=status,
                        n_materiales=len(stored),error_max_abs=err,nota='baseline + sum(lambda_j*t_target_j)'))
    teacher=t
except Exception as ex:
    replays.append(dict(arquitectura='TEACHER_R3',split='ARCHIVO_COMPLETO',estado='PENDIENTE',
                        n_materiales='',error_max_abs='',nota=str(ex)))

# ================= REPLAY DE H2B2A7-13: SAFE + QWC24/64 =============
# Se reproduce la EVALUACION ALMACENADA, no se vuelve a entrenar ni se altera OUTER HOLD.
can_replay=bool(REPLAY_ARCHIVED_OUTER and manifest_status in ('COINCIDE','NO_DISPONIBLE','SIN_HASHES_COMPARABLES')
                and all(lookup.get(k) for k in ['TEACHER_R3','HEAD4','SAFE_RANK2','QWC24','QWC24_CONTROL',
                                            'QWC64','QWC_ORIG','X6_PAULI','OUTER_SCORES']))
if manifest_status=='NO_DISPONIBLE':
    print('NOTA: No hay MANIFEST_PREVIO; se permite cotejo contra CSV archivado, NO se afirma autenticacion previa.')
if not can_replay:
    replays.append(dict(arquitectura='SAFE_QWC24_QWC64',split='OUTER_HOLD_YA_ARCHIVADO',estado='PENDIENTE',
                        n_materiales='',error_max_abs='',nota='Faltan fuentes o hay divergencia SHA; ningun score inventado.'))
else:
    try:
        s=get_npz('SAFE_RANK2',EXPECTED_KEYS['SAFE_RANK2'])
        h=get_npz('HEAD4',EXPECTED_KEYS['HEAD4'])
        c24=get_npz('QWC24',EXPECTED_KEYS['QWC24'])
        c64=get_npz('QWC64',EXPECTED_KEYS['QWC64'])
        c10=get_npz('QWC24_CONTROL',EXPECTED_KEYS['QWC24_CONTROL'])
        orig=get_npz('QWC_ORIG',EXPECTED_KEYS['QWC_ORIG'])
        t=t if teacher is not None else get_npz('TEACHER_R3',EXPECTED_KEYS['TEACHER_R3'])
        X6=np.load(lookup['X6_PAULI'],mmap_mode='r',allow_pickle=False)
        ho=np.asarray(t['outer_hold_idx'],int).ravel()
        tr=np.asarray(t['inner_train_idx'],int).ravel()
        va=np.asarray(t['inner_val_idx'],int).ravel()
        L=np.asarray(t['lambdas'],float);base0=np.asarray(t['baseline'],float).ravel()
        W=np.asarray(s['W'],float);d=np.asarray(s['d'],float).ravel()
        B=np.asarray(s['B'],float);C=np.asarray(s['C'],float)
        A4=np.asarray(h['A4'],float);c4=np.asarray(h['c4'],float).ravel()
        head_coeff=np.asarray(h['head_coefficients'],float)
        cp=np.asarray(orig['coef_pauli_original'],float)
        n=len(base0)
        assert X6.shape==(n,4096) and L.shape==(n,3)
        assert W.shape==(4095,3) and B.shape==(4095,2) and C.shape==(2,3)
        assert A4.shape==(3,4) and head_coeff.shape==(4,4096) and cp.shape==(2,4096)
        assert len(set(ho))==len(ho) and set(ho).isdisjoint(set(tr)) and set(ho).isdisjoint(set(va))
        assert np.array_equal(np.asarray(c24['global_val']).ravel(),va)
        assert np.array_equal(np.asarray(c64['global_val']).ravel(),va)
        for key in ['theta','basis_ids','shots_por_base']:
            assert np.array_equal(c24[key],c10[key]),f'QWC24 no coincide con control: {key}'
        assert np.array_equal(c24['threshold_fit'],c10['threshold_fit'])
        assert len(c24['basis_ids'])==24 and len(c64['basis_ids'])==64
        assert int(np.sum(c24['shots_por_base']))==10000 and int(np.sum(c64['shots_per_base']))==10000
        BASIS=np.asarray(list(product((1,2,3),repeat=6)),dtype=np.int8)
        def all_paulis(axes):
            terms=[]
            for mask in range(64):
                xx=zz=0
                for q in range(6):
                    if (mask>>q)&1:
                        if axes[q] in (1,2):xx|=1<<q
                        if axes[q] in (2,3):zz|=1<<q
                terms.append(64*xx+zz)
            return np.asarray(terms,int)
        def unpack(ck):
            bids=np.asarray(ck['basis_ids'],int).ravel()
            idx=np.asarray(ck['pauli_index'],int)
            theta=np.asarray(ck['theta'],float).ravel()
            result=[];off=0;seen=set()
            assert np.all((0<=bids)&(bids<729)) and len(set(bids))==len(bids)
            for ib,bid in enumerate(bids):
                sub=idx[idx[:,0]==ib]
                ks=sub[sub[:,1]==0,2];size=len(ks)
                assert size>0 and len(sub)==2*size
                assert np.array_equal(sub[:,1],np.r_[np.zeros(size,int),np.ones(size,int)])
                assert np.array_equal(sub[size:,2],ks)
                assert not set(map(int,ks))&seen
                seen.update(map(int,ks))
                assert set(map(int,ks)).issubset(set(map(int,all_paulis(BASIS[int(bid)]))))
                result.append((ks,theta[off:off+size],theta[off+size:off+2*size]))
                off+=2*size
            assert off==len(theta)
            return result
        groups24=unpack(c24);groups64=unpack(c64)
        def calculate(ix):
            ix=np.asarray(ix,int)
            x=np.asarray(X6[ix],float)
            z4=x@head_coeff.T
            t4=z4@A4.T+c4
            residual=x[:,1:]@W+d
            latent=t4+residual
            znew=x[:,1:]@B
            ll=L[ix]
            base=base0[ix]+np.einsum('ij,ij->i',ll,latent-znew@C)+(ll@C.T)@cp[:,0]
            safe=base0[ix]+np.einsum('ij,ij->i',ll,latent)
            weights=ll@C.T
            def readout(groups):
                r=np.asarray(base,float).copy()
                for inds,left,right in groups:
                    xx=np.asarray(X6[np.ix_(ix,inds)],float)
                    r+=(xx@left)*weights[:,0]+(xx@right)*weights[:,1]
                return r
            return z4,latent,znew,safe,readout(groups24),readout(groups64)
        # Comparacion previa con FIT/VAL, sin calcular nuevas metricas sobre HOLD.
        for idx,fold in [(tr,'TRAIN'),(va,'INNER_VAL')]:
            z4,lat,znew,_,_,_=calculate(idx)
            want_z=h['Z4_train'] if fold=='TRAIN' else h['Z4_val']
            want_lat=s['T_train'] if fold=='TRAIN' else s['T_val']
            want_new=s['znew_train'] if fold=='TRAIN' else s['znew_val']
            max_err=max(float(np.max(abs(z4-want_z))),float(np.max(abs(lat-want_lat))),
                        float(np.max(abs(znew-want_new))))
            if max_err>=2e-3:raise RuntimeError(f'{fold}: checkpoints internos incompatibles, error {max_err}')
            replays.append(dict(arquitectura='HEAD4_SAFE',split=fold,estado='REPRODUCIDO',
                                n_materiales=len(idx),error_max_abs=max_err,nota='Reconstruccion congelada 13, tolerancia 2e-3'))
        stored_rows=csv_read(lookup['OUTER_SCORES'])
        required=['global_index','score_SAFE_ideal','score_QWC24_ideal','score_QWC64_ideal','y_FM']
        if not stored_rows or any(a not in stored_rows[0] for a in required):
            raise RuntimeError('CSV archivado OUTER sin columnas requeridas. No se reevalua sin el archivo.')
        archived={int(r['global_index']):r for r in stored_rows}
        if len(archived)!=len(stored_rows) or set(archived)!=set(map(int,ho)):
            raise RuntimeError('Indices de OUTER no coinciden exactamente con CSV archivado.')
        z4,lat,znew,safe,replay24,replay64=calculate(ho)
        yy=np.asarray(t['y']).ravel()
        if set(np.unique(yy))=={-1,1}:yy=(yy==1).astype(int)
        elif set(np.unique(yy))=={0,1}:yy=yy.astype(int)
        else:raise RuntimeError('Etiquetas teacher no binarias')
        results={'SAFE_ideal':safe,'QWC24_ideal':replay24,'QWC64_ideal':replay64}
        err_by={k:[] for k in results}
        for j,gi in enumerate(ho):
            old=archived[int(gi)]
            if int(old['y_FM'])!=int(yy[gi]):raise RuntimeError(f'Etiquetas no coinciden para global_index {gi}')
            row={'global_index':int(gi),'y_FM_ARCHIVADA':int(old['y_FM']),
                 'teacher_s3_archivado_checkpoint':float(t['s3'][gi])}
            for key,vector in results.items():
                col='score_'+key
                v=float(vector[j]);a=float(old[col]);err=abs(v-a)
                row['score_'+key+'_replay']=v
                row['score_'+key+'_archivado']=a
                row['error_'+key]=err
                err_by[key].append(err)
            score_rows.append(row)
        for key,values in err_by.items():
            e=max(values)
            replays.append(dict(arquitectura=key,split='OUTER_HOLD_ARCHIVADO',
                                estado='REPRODUCIDO' if e<=TOL_SCORE_MAX else 'DIVERGENCIA_NUMERICA',
                                n_materiales=len(ho),error_max_abs=e,nota='Mismos indices y etiquetas, no reentrenamiento'))
        replay_details={'max_err':{k:max(v) for k,v in err_by.items()},
                        'n_outer':len(ho),'n_qwc24_bases':24,'n_qwc64_bases':64}
    except Exception as ex:
        read_errors.append('REPLAY_H2B2A7_13: '+str(ex))
        replays.append(dict(arquitectura='SAFE_QWC24_QWC64',split='OUTER_HOLD_ARCHIVADO',
                            estado='ERROR_AUDITORIA',n_materiales='',error_max_abs='',nota=str(ex)))
        score_rows=[]

# ================== AUDITORIA STUDENT (SIN SIMULACION NULA) ==================
# Un QPY congelado y los parametros son evidencia de un decodificador, pero
# NO proporcionan por si solos score por material sin rho6 y el programa de lectura.
student_status='NO_ENCONTRADO';student_info={}
if lookup['STUDENT_COMP']:
    try:
        model=get_npz('STUDENT_COMP',EXPECTED_KEYS['STUDENT_COMP'])
        angles=np.asarray(model['rxx_angles'])
        student_status='PARAMETROS_ENCONTRADOS_SCORE_NO_REPRODUCIDO'
        student_info={'rxx_angles_shape':list(angles.shape),'rxx_angles_numel':int(angles.size),
                      'diagonal_shape':list(np.asarray(model['diagonal']).shape),
                      'qpy_disponible':bool(lookup['STUDENT_COMP_QPY'])}
    except Exception as ex:
        student_status='PARAMETROS_INCOMPATIBLES';read_errors.append('STUDENT_COMP: '+str(ex))
replays.append(dict(arquitectura='STUDENT_U_DEC_D',split='INNER_VAL / OUTER',estado=student_status,
                    n_materiales='',error_max_abs='',nota='Pendiente de reconstruir amplitudes/mezclas y circuitos U,D; no equivale a SAFE/QWC'))

# ==================== AUDITORIA DE COSTOS POR MATERIAL =======================
# Clasificamos COSTO TOTAL vs INCREMENTAL/INCOMPLETO; no sumamos cifras sin soporte.
costs=[]
costs.append(dict(ruta='QWC24_ARCHIVADA',lecturas_bases=24,shots_rho6_incrementales=10000,
                  shots_rho4='',shots_totales_plan_logico='',tipo='INCREMENTAL_PARCIAL',
                  evidencia='H2B2A7-13: 10k shots solo lectura QWC24; 4 cabezales y SAFE residual ideales',
                  advertencia='NO es costo completo por material'))
costs.append(dict(ruta='QWC64_ARCHIVADA',lecturas_bases=64,shots_rho6_incrementales=10000,
                  shots_rho4='',shots_totales_plan_logico='',tipo='INCREMENTAL_PARCIAL',
                  evidencia='H2B2A7-13: 10k shots solo lectura QWC64',
                  advertencia='NO es costo completo por material'))
if lookup['QWC_SHOTS_REFERENCE']:
    try:
        alloc=get_npz('QWC_SHOTS_REFERENCE',EXPECTED_KEYS['QWC_SHOTS_REFERENCE'])
        bids=np.asarray(alloc['basis_ids'],int)
        n6=np.asarray(alloc['n6_por_base'],int)
        n4=np.asarray(alloc['n4_por_canal'],int)
        weights=np.asarray(alloc['costos_copias'],int)
        if n6.ndim!=1 or len(n6)!=len(bids) or len(n4)!=len(weights):raise ValueError('Arreglos de asignacion incompatibles')
        if np.any(n6<0) or np.any(n4<0) or np.any(weights<1):raise ValueError('Asignaciones no fisicas')
        total6=int(n6.sum());total4=int(n4@weights)
        costs.append(dict(ruta='PAULI_QWC_COMPLETO_16_REFERENCIA',lecturas_bases=int(len(bids)),
                  shots_rho6_incrementales=total6,shots_rho4=total4,
                  shots_totales_plan_logico=total6+total4,
                  tipo='PRESUPUESTO_LOGICO_DE_MEDICION_REFERENCIA',
                  evidencia=f'Asignacion H2B2A7-16: {len(bids)} bases; rho4 canales ponderados por costo de copias',
                  advertencia='No es tiempo QPU ni costo fisico total: requiere verificar preparaciones/ramas/layout/overhead'))
    except Exception as ex:read_errors.append('COSTO_H2B2A7_16: '+str(ex))
else:
    costs.append(dict(ruta='PAULI_QWC_COMPLETO_16_REFERENCIA',lecturas_bases='',
                  shots_rho6_incrementales='',shots_rho4='',shots_totales_plan_logico='',
                  tipo='PENDIENTE_CHECKPOINT',evidencia='asignacion_referencia_100k_10k.npz faltante',
                  advertencia='No sustituir por 24 ni 64 bases'))
if lookup['QWC365_COVERAGE']:
    try:
        cov=json.loads(lookup['QWC365_COVERAGE'].read_text(encoding='utf-8'))
        # El certificado contiene la cobertura y sus presupuestos, no garantiza el mapping fisico.
        (OUT/'INTEGRACION00_CERTIFICADO_COBERTURA_ORIGINAL_REFERENCIA.json').write_text(
            json.dumps({'archivo':str(lookup['QWC365_COVERAGE']),'sha256':sha256(lookup['QWC365_COVERAGE']),
                        'claves_raiz':sorted(cov.keys()) if isinstance(cov,dict) else [],
                        'nota':'Se referencia el archivo original; no se copia ni se recalcula el MILP.'},
                       indent=2,ensure_ascii=False),encoding='utf8')
    except Exception as ex:read_errors.append('CERTIFICADO_COBERTURA: '+str(ex))
costs.append(dict(ruta='STUDENT_DOS_DECODIFICADORES',lecturas_bases='2 cabezales de correccion (parcial)',
                  shots_rho6_incrementales='',shots_rho4='',shots_totales_plan_logico='',
                  tipo='NO_CUANTIFICADO_TOTAL',
                  evidencia='H2B2A5-1E/1F: parametros U,D para correccion SAFE rank-2',
                  advertencia='Incluye otros 4 cabezales, mezclas, rho4 y shot allocation por definir; RXX no son shots'))

# ================== INFORME, BITACORA E INTEGRIDAD =========================
csv_save(OUT/'INTEGRACION00_CHECKPOINTS.csv',checks)
csv_save(OUT/'INTEGRACION00_REPRODUCIBILIDAD.csv',replays)
csv_save(OUT/'INTEGRACION00_COSTOS_POR_MATERIAL.csv',costs)
if score_rows:
    csv_save(OUT/'INTEGRACION00_SCORES_OUTER_REPLAY_ARCHIVADO.csv',score_rows)

mandatory=['TEACHER_R3','HEAD4','SAFE_RANK2','X6_PAULI','QWC24','QWC24_CONTROL','QWC64','QWC_ORIG','OUTER_SCORES']
missing_mandatory=[k for k in mandatory if not lookup.get(k)]
confirmed=all(r['estado']=='REPRODUCIDO' for r in replays if r['arquitectura'] in ('SAFE_ideal','QWC24_ideal','QWC64_ideal'))
confirmed=bool(confirmed and any(r['arquitectura']=='QWC24_ideal' for r in replays))
confirmed=confirmed and any(r['arquitectura']=='TEACHER_R3' and r['estado']=='REPRODUCIDO' for r in replays)
status=('REPLAY_PAULI_QWC_VERIFICADO_STUDENT_PENDIENTE'
        if confirmed and manifest_status=='COINCIDE' and not read_errors and not manifest_mismatches
        else ('REPLAY_COTEJADO_SIN_CERTIFICACION_MANIFIESTO'
              if confirmed and not read_errors and not manifest_mismatches
              else 'AUDITORIA_PARCIAL_O_PENDIENTE_ARCHIVOS'))

summary={
  'experimento':VERSION,'fecha_utc':datetime.now(timezone.utc).isoformat(),
  'root':str(ROOT),'carpeta_salida':str(OUT),
  'estado':status,'sin_qiskit':True,'sin_api_ibm':True,'sin_qpu_shots':True,
  'sin_entrenamiento':True,'sin_seleccion_de_modelo_outer':True,
  'out_hold_previamente_abierto':True,
  'integridad_manifiesto_previo_13':manifest_status,
  'divergencias_manifiesto':manifest_mismatches,
  'checkpoint_localizados':sum(bool(v) for v in lookup.values()),
  'checkpoint_catalogo_total':len(lookup),
  'faltantes_minimos_para_replay':missing_mandatory,
  'replay':replays,'student_estado':student_status,'student_parametros':student_info,
  'replay_detalles':replay_details,
  'costos':costs,'errores_y_reservas':read_errors,
  'no_comparar_inner_val_con_outer_hold_como_test_pareado':True,
  'nota_end_to_end':('Esta auditoria NO prepara estados ni ejecuta U_dec fisicamente. '
     'Los presupuestos del estudio 13 de 10k shots son parciales; los modelos del student '
     'no tienen costo completo certificado y su score solo se declara reproducido si hay auditoria equivalente.'),
}
json_save(OUT/'INTEGRACION00_RESUMEN.json',summary)

report=f'''# {VERSION} — Auditoría unificada de las arquitecturas

**Estado:** `{status}`. Fecha UTC: {summary['fecha_utc']}.

## 1. Propósito y protección de la evaluación

- **No** se entrena, reoptimiza, transpila, consulta IBM o envía trabajo.
- Los scores OUTER HOLD en H2B2A7-13 **ya fueron evaluados**; aquí solamente se compara
  una reconstrucción independiente del score con la predicción previamente archivada.
- **No** seleccionar un nuevo modelo utilizando OUTER HOLD para volver a llamarlo validación independiente.
- Las métricas INNER VAL de teacher/student y las métricas OUTER HOLD de SAFE/QWC no
  son comparables como diferencias pareadas sin un split idéntico.

## 2. Procedencia y checkpoints

- Encontrados: **{summary['checkpoint_localizados']}/{summary['checkpoint_catalogo_total']}**.
- Manifiesto H2B2A7-13 previo a OUTER: **{manifest_status}**.
- Checkpoints mínimos faltantes: `{missing_mandatory}`.
- Los archivos se identifican por nombre, ruta y SHA-256; una mera coincidencia de nombre
  no autentica por sí misma un checkpoint si falta su manifiesto original.

## 3. Auditoría de score

- Reconstrucción teacher: `baseline + suma(lambdas * t_target)`.
- Reconstrucción SAFE/QWC24/QWC64: fórmulas exactas del código H2B2A7-13,
  si existen los siete checkpoints originales, X6 y `predicciones_OUTER_HOLD.csv`.
- El replay verifica los índices y las etiquetas **sin reajustar pesos ni umbrales**.
- Student: **{student_status}**. Un QPY y sus parámetros no equivalen a scores
  reconstruidos para materiales sin preparación real de estados y decodificación de bits.

## 4. Costo completo vs incremental

La evaluación QWC24/QWC64 del H2B2A7-13 asignó 10 000 shots a la
parte incremental QWC. Las cuatro cabezas originales, el residual SAFE y las
características multicopia se trataron idealmente, por tanto no equivale al
costo completo por material.

La referencia H2B2A7-16, si existe su checkpoint, contiene una asignación
física-lógica de shots a las bases rho6 y copias rho4; se reporta por separado.
**No** convertir shots de subrutinas a segundos reales QPU sin circuitos ISA,
mezclas y tiempos de backend. No contar dos veces canales compartidos.

La corrección student rank-2 requiere considerar los dos decodificadores U,D,
los cuatro cabezales previos, los estados de entrada y las características rho4.
Dos decodificadores **no** equivalen a dos shots ni necesariamente a dos circuitos
por material si la preparación involucra mezcla de componentes.

## 5. Siguiente decisión metodológica

1. Completar archivos faltantes y resolver duplicados de SHA.
2. Certificar replay de scores y alinear particiones por `global_index`.
3. Auditar score de student en INNER VAL, sin usar OUTER para entrenamiento.
4. Obtener costo total por material, no solo número de bases o RXX.
5. Producir una comparación A/B de dos implementaciones congeladas en un split común.
6. Solo después preparar un piloto end-to-end y su autorización independiente.

## 6. Errores o reservas

{os.linesep.join('- '+v for v in read_errors) if read_errors else '- No hay fallos detectados en las comprobaciones ejecutadas.'}
'''
(OUT/'INTEGRACION00_INFORME_CIENTIFICO.md').write_text(report,encoding='utf8')

results={f.name:sha256(f) for f in sorted(OUT.iterdir()) if f.is_file()}
json_save(OUT/'INTEGRACION00_HUELLAS_SHA256.json',{
    'experimento':VERSION,'productos_sha256':results,
    'fuentes':{k:{'ruta':str(p),'sha256':sha256(p)} for k,p in lookup.items() if p},
    'sin_envios_ibm':True,'sin_reentrenamiento':True,
    'manifiesto_no_autoreferenciado':True,
})
print('\n'+'='*99)
print('INTEGRACION-00 |',status)
print('Encontrados:',summary['checkpoint_localizados'],'/',summary['checkpoint_catalogo_total'])
print('Manifiesto pre-OUTER:',manifest_status)
print('Replay:',[(r['arquitectura'],r['estado']) for r in replays])
print('Student:',student_status)
print('Archivos generados:',', '.join(f.name for f in sorted(OUT.iterdir()) if f.is_file()))
print('Sin IBM, sin entrenamiento, sin nuevos experimentos. Directorio:',OUT)
print('='*99)
